# Sales Funnel Drop-Off Analysis

## Project Overview
This notebook analyses a multi-stage marketing and sales funnel to identify where users drop off, how conversion rates differ across channels, regions, devices, and cohorts, and which transitions create the greatest revenue leakage.

## Learning Objectives
- Quantify funnel-stage retention and drop-off clearly.
- Compare conversion outcomes across acquisition and customer segments.
- Identify where funnel leakage is concentrated and which cohorts perform best.
- Translate funnel diagnostics into practical optimisation priorities.

## Business / Research Problem
Teams need to know exactly where leads are failing to progress through the funnel, which segments underperform, and which stage improvements would recover the most revenue.

## Why This Analysis Matters
A structured funnel analysis helps marketing and sales teams prioritise spend, fix friction points, and target conversion improvements where they are most likely to change business outcomes.

## Dataset Overview

**Source**: Synthetic dataset generated inside the notebook (seed = 42 for reproducibility).

**Schema**

| Column | Type | Description |
|---|---|---|
| lead_id | int | Unique lead identifier |
| cohort_month | str | Month the lead entered the funnel (2024-01 to 2024-12) |
| channel | str | Acquisition channel: Organic / Paid / Email / Referral |
| region | str | Sales region: North / South / East / West |
| device | str | Session device: Mobile / Desktop / Tablet |
| funnel_stage | int | Last completed stage (0–4): 0=Awareness, 1=Interest, 2=Consideration, 3=Intent, 4=Purchase |
| stage_label | str | Human-readable stage name |
| converted | bool | True if lead reached the Purchase stage |
| revenue | float | Revenue generated (>0 only for converted leads) |

**Funnel stages**: Awareness → Interest → Consideration → Intent → Purchase

## Dataset Source and License Notes
This notebook generates a synthetic dataset inside the notebook itself for learning and reproducibility. Because the data is simulated, there are no external licensing constraints, but the outputs should be interpreted as a teaching example rather than real operational evidence.

## Environment Setup

In [ ]:
import subprocess, sys

required = ["pandas", "numpy", "matplotlib", "seaborn"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages available.")

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110

print("Imports complete.")

## Configuration / Constants

In [ ]:
SEED = 42
rng  = np.random.default_rng(SEED)

STAGES = ["Awareness", "Interest", "Consideration", "Intent", "Purchase"]
N_STAGES = len(STAGES)

CHANNELS = ["Organic", "Paid", "Email", "Referral"]
REGIONS  = ["North", "South", "East", "West"]
DEVICES  = ["Mobile", "Desktop", "Tablet"]

# Cohorts: monthly waves Jan-2024 through Dec-2024
import pandas as _pd
COHORTS = [f"2024-{m:02d}" for m in range(1, 13)]

# Approximate lead volumes per cohort (growing trend + noise)
COHORT_LEADS = {c: int(500 + i*30 + rng.integers(-40, 40)) for i, c in enumerate(COHORTS)}

# Per-stage drop-off probabilities (prob of advancing to next stage) by channel
# Organic: best-intent traffic; Paid: volume but weaker; Email: moderate; Referral: highest intent
CHANNEL_CONV = {
    "Organic":  [0.68, 0.52, 0.44, 0.55],   # Awareness->Interest, Interest->Consid, Consid->Intent, Intent->Purchase
    "Paid":     [0.48, 0.36, 0.28, 0.42],
    "Email":    [0.60, 0.46, 0.38, 0.50],
    "Referral": [0.72, 0.60, 0.52, 0.64],
}

# Regional multipliers (applied on top of channel rates)
REGION_MULT = {"North": 1.05, "South": 0.94, "East": 1.00, "West": 0.98}

# Device multipliers
DEVICE_MULT = {"Desktop": 1.08, "Mobile": 0.90, "Tablet": 0.97}

# Revenue distribution for converted leads (per channel)
REVENUE_PARAMS = {"Organic": (220, 60), "Paid": (180, 50), "Email": (200, 55), "Referral": (260, 70)}

print("Configuration ready.")
print(f"Cohorts: {COHORTS[0]} → {COHORTS[-1]}")
print(f"Stages: {' → '.join(STAGES)}")

## Dataset Generation

In [ ]:
rows = []

for cohort in COHORTS:
    n_leads = COHORT_LEADS[cohort]
    channels = rng.choice(CHANNELS, size=n_leads, p=[0.30, 0.35, 0.20, 0.15])
    regions  = rng.choice(REGIONS,  size=n_leads, p=[0.28, 0.22, 0.28, 0.22])
    devices  = rng.choice(DEVICES,  size=n_leads, p=[0.52, 0.38, 0.10])

    for ch, reg, dev in zip(channels, regions, devices):
        conv_probs = CHANNEL_CONV[ch]
        rm = REGION_MULT[reg]
        dm = DEVICE_MULT[dev]

        stage = 0   # everyone enters at Awareness
        for s in range(N_STAGES - 1):   # can advance from stage 0..3 to 1..4
            p = min(conv_probs[s] * rm * dm, 0.96)
            if rng.random() < p:
                stage += 1
            else:
                break

        converted = stage == 4
        if converted:
            mu, sigma = REVENUE_PARAMS[ch]
            revenue = float(rng.normal(mu, sigma))
            revenue = max(revenue, 20.0)
        else:
            revenue = 0.0

        rows.append({
            "cohort_month": cohort,
            "channel": ch,
            "region": reg,
            "device": dev,
            "funnel_stage": stage,
            "stage_label": STAGES[stage],
            "converted": converted,
            "revenue": revenue,
        })

df = pd.DataFrame(rows)
df.insert(0, "lead_id", range(1, len(df) + 1))

print(f"Dataset shape: {df.shape}")
print(f"Total leads: {len(df):,}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

## Data Validation Checks

In [ ]:
print("=== Data Quality Check ===")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nValue counts for funnel_stage:")
print(df["funnel_stage"].value_counts().sort_index())
print(f"\nChannel distribution:")
print(df["channel"].value_counts())
print(f"\nRegion distribution:")
print(df["region"].value_counts())
print(f"\nDevice distribution:")
print(df["device"].value_counts())
print(f"\nConverted rate: {df['converted'].mean():.2%}")
print(f"Total revenue (converted leads): ${df['revenue'].sum():,.0f}")

## Data Cleaning

Because this funnel dataset is generated in-notebook, cleaning is lightweight. We still standardise types explicitly so later calculations and plots depend on audited schema rather than implicit pandas casting.

In [ ]:
# cohort_month is already a string (e.g. '2022-01') — just ensure consistent format
df['cohort_month'] = df['cohort_month'].astype(str).str[:7]
for col in ['channel', 'region', 'device', 'stage_label']:
    df[col] = df[col].astype('category')
df['converted'] = df['converted'].astype(bool)
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0.0)
print(df.dtypes[['cohort_month', 'channel', 'region', 'device',
                 'stage_label', 'converted', 'revenue']])

## Exploratory Data Analysis

In [ ]:
# Build stage-level summary
# For each stage, count leads that reached at LEAST that stage
stage_counts = {}
for s, label in enumerate(STAGES):
    stage_counts[label] = (df["funnel_stage"] >= s).sum()

funnel_summary = pd.DataFrame({
    "Stage": STAGES,
    "Stage_Index": range(N_STAGES),
    "Leads_Reached": [stage_counts[s] for s in STAGES],
})

funnel_summary["Drop_From_Previous"] = funnel_summary["Leads_Reached"].diff().abs().fillna(0).astype(int)
funnel_summary["Retention_Rate"]     = funnel_summary["Leads_Reached"] / funnel_summary["Leads_Reached"].iloc[0]
funnel_summary["Stage_Conversion"]   = funnel_summary["Leads_Reached"] / funnel_summary["Leads_Reached"].shift(1)
funnel_summary["Stage_DropOff_Rate"] = 1 - funnel_summary["Stage_Conversion"]

print("=== Funnel Overview ===")
print(funnel_summary[["Stage","Leads_Reached","Drop_From_Previous","Retention_Rate","Stage_Conversion","Stage_DropOff_Rate"]].to_string(index=False))

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute counts
colors = sns.color_palette("Blues_d", N_STAGES)[::-1]
axes[0].barh(funnel_summary["Stage"][::-1], funnel_summary["Leads_Reached"][::-1], color=colors)
for idx, row in funnel_summary[::-1].reset_index(drop=True).iterrows():
    axes[0].text(row["Leads_Reached"] + 20, idx, f'{row["Leads_Reached"]:,}', va='center', fontsize=9)
axes[0].set_xlabel("Leads Reached Stage")
axes[0].set_title("Funnel : Leads by Stage")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: retention %
bar_colors = [sns.color_palette("RdYlGn", 10)[int(r * 9)] for r in funnel_summary["Retention_Rate"]]
bars = axes[1].bar(funnel_summary["Stage"], funnel_summary["Retention_Rate"] * 100, color=bar_colors, edgecolor='white')
for bar, rate in zip(bars, funnel_summary["Retention_Rate"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{rate:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_ylabel("% of Initial Leads Retained")
axes[1].set_title("Funnel Retention Rate by Stage")
axes[1].set_ylim(0, 115)

plt.tight_layout()
plt.savefig("funnel_overview.png", bbox_inches='tight', dpi=110)
plt.show()
print("Saved: funnel_overview.png")

## Bivariate / Multivariate Analysis

In [ ]:
# Identify highest-leakage stage (by absolute drop AND by relative drop-off rate)
valid_rows = funnel_summary.iloc[1:].copy()   # skip Awareness (no previous stage)

abs_worst  = valid_rows.loc[valid_rows["Drop_From_Previous"].idxmax(), "Stage"]
rate_worst = valid_rows.loc[valid_rows["Stage_DropOff_Rate"].idxmax(), "Stage"]

print(f"Stage with largest ABSOLUTE drop:  {abs_worst}")
print(f"Stage with highest DROP-OFF RATE:   {rate_worst}")
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Absolute drop
palette = ["#e74c3c" if s in [abs_worst] else "#85c1e9" for s in funnel_summary["Stage"].iloc[1:]]
axes[0].bar(funnel_summary["Stage"].iloc[1:], funnel_summary["Drop_From_Previous"].iloc[1:], color=palette, edgecolor='white')
axes[0].set_title("Leads Lost per Stage Transition")
axes[0].set_ylabel("Leads Dropped")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(axes[0].patches, funnel_summary["Drop_From_Previous"].iloc[1:]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{int(val):,}', ha='center', fontsize=9)

# Drop-off rate
palette2 = ["#e74c3c" if s in [rate_worst] else "#85c1e9" for s in funnel_summary["Stage"].iloc[1:]]
axes[1].bar(funnel_summary["Stage"].iloc[1:], funnel_summary["Stage_DropOff_Rate"].iloc[1:] * 100, color=palette2, edgecolor='white')
axes[1].set_title("Stage Drop-Off Rate (%)")
axes[1].set_ylabel("% of Entering Leads Lost")
for bar, val in zip(axes[1].patches, funnel_summary["Stage_DropOff_Rate"].iloc[1:]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("stage_dropoff.png", bbox_inches='tight', dpi=110)
plt.show()

## Feature-Specific Insights

In [ ]:
def conv_rate_table(frame, col):
    tbl = frame.groupby(col).agg(
        Leads=("lead_id", "count"),
        Converted=("converted", "sum"),
        Revenue=("revenue", "sum"),
    ).reset_index()
    tbl["Conversion_Rate"] = tbl["Converted"] / tbl["Leads"]
    tbl["Avg_Revenue_Per_Lead"] = tbl["Revenue"] / tbl["Leads"]
    return tbl.sort_values("Conversion_Rate", ascending=False)

channel_conv = conv_rate_table(df, "channel")
print("=== Conversion Rate by Channel ===")
print(channel_conv.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pal = sns.color_palette("Set2", len(channel_conv))

axes[0].barh(channel_conv["channel"], channel_conv["Conversion_Rate"]*100, color=pal)
axes[0].set_xlabel("Overall Conversion Rate (%)")
axes[0].set_title("Conversion Rate by Channel")
for bar, v in zip(axes[0].patches, channel_conv["Conversion_Rate"]):
    axes[0].text(v*100 + 0.1, bar.get_y() + bar.get_height()/2, f'{v:.1%}', va='center', fontsize=9)

axes[1].barh(channel_conv["channel"], channel_conv["Avg_Revenue_Per_Lead"], color=pal)
axes[1].set_xlabel("Avg Revenue per Lead ($)")
axes[1].set_title("Revenue Efficiency by Channel")
for bar, v in zip(axes[1].patches, channel_conv["Avg_Revenue_Per_Lead"]):
    axes[1].text(v + 0.3, bar.get_y() + bar.get_height()/2, f'${v:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig("channel_conversion.png", bbox_inches='tight', dpi=110)
plt.show()

## Visualization 3: Stage Progression Heatmap by Channel

In [ ]:
# For each channel, what % of that channel's leads reached each stage?
stage_heat = pd.DataFrame()
for ch in CHANNELS:
    sub = df[df["channel"] == ch]
    total = len(sub)
    row = {s: (sub["funnel_stage"] >= idx).sum() / total for idx, s in enumerate(STAGES)}
    row["channel"] = ch
    stage_heat = pd.concat([stage_heat, pd.DataFrame([row])], ignore_index=True)

stage_heat = stage_heat.set_index("channel")[STAGES]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(stage_heat * 100, annot=True, fmt=".1f", cmap="YlOrRd_r",
            linewidths=0.5, ax=ax, cbar_kws={"label": "% Reached Stage"})
ax.set_title("Stage Retention Heatmap by Channel (%)")
ax.set_xlabel("Funnel Stage")
ax.set_ylabel("Channel")
plt.tight_layout()
plt.savefig("heatmap_channel.png", bbox_inches='tight', dpi=110)
plt.show()

## Segment Analysis: Conversion Rate by Region

In [ ]:
region_conv = conv_rate_table(df, "region")
print("=== Conversion Rate by Region ===")
print(region_conv.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette("pastel", len(region_conv))
bars = ax.bar(region_conv["region"], region_conv["Conversion_Rate"]*100, color=colors, edgecolor='white')
for bar, v in zip(bars, region_conv["Conversion_Rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel("Conversion Rate (%)")
ax.set_title("Overall Funnel Conversion Rate by Region")
plt.tight_layout()
plt.savefig("region_conversion.png", bbox_inches='tight', dpi=110)
plt.show()

## Segment Analysis: Conversion Rate by Device

In [ ]:
device_conv = conv_rate_table(df, "device")
print("=== Conversion Rate by Device ===")
print(device_conv.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2ecc71" if d == "Desktop" else "#e74c3c" if d == "Mobile" else "#f39c12" for d in device_conv["device"]]
bars = ax.bar(device_conv["device"], device_conv["Conversion_Rate"]*100, color=colors, edgecolor='white')
for bar, v in zip(bars, device_conv["Conversion_Rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel("Conversion Rate (%)")
ax.set_title("Conversion Rate by Device Type")
plt.tight_layout()
plt.savefig("device_conversion.png", bbox_inches='tight', dpi=110)
plt.show()

## Cross-Segment Analysis: Channel × Region Heatmap

In [ ]:
cross = df.groupby(["channel", "region"]).agg(
    Leads=("lead_id","count"),
    Converted=("converted","sum")
).reset_index()
cross["Conv_Rate"] = cross["Converted"] / cross["Leads"]

pivot = cross.pivot(index="channel", columns="region", values="Conv_Rate") * 100

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Blues", linewidths=0.5, ax=ax,
            cbar_kws={"label": "Conversion Rate (%)"})
ax.set_title("Conversion Rate (%) — Channel × Region")
plt.tight_layout()
plt.savefig("cross_segment_heatmap.png", bbox_inches='tight', dpi=110)
plt.show()

## Cohort Analysis: Monthly Entry Cohorts

In [ ]:
cohort_stats = df.groupby("cohort_month").agg(
    Leads=("lead_id","count"),
    Converted=("converted","sum"),
    Revenue=("revenue","sum"),
).reset_index()
cohort_stats["Conversion_Rate"] = cohort_stats["Converted"] / cohort_stats["Leads"]
cohort_stats["Avg_Revenue_Per_Converted"] = cohort_stats["Revenue"] / cohort_stats["Converted"].replace(0, np.nan)

print("=== Cohort Conversion Summary ===")
print(cohort_stats.to_string(index=False))

## Visualization: Cohort Conversion Rate Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Top: leads volume
axes[0].bar(cohort_stats["cohort_month"], cohort_stats["Leads"], color="#85c1e9", edgecolor='white', label="Leads")
axes[0].bar(cohort_stats["cohort_month"], cohort_stats["Converted"], color="#2ecc71", edgecolor='white', label="Converted")
axes[0].set_ylabel("Count")
axes[0].set_title("Monthly Cohort: Leads vs. Converted")
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Bottom: conversion rate trend
axes[1].plot(cohort_stats["cohort_month"], cohort_stats["Conversion_Rate"]*100,
             marker='o', color="#e74c3c", linewidth=2, markersize=7)
for x, y in zip(cohort_stats["cohort_month"], cohort_stats["Conversion_Rate"]):
    axes[1].annotate(f'{y:.1%}', (x, y*100), textcoords="offset points", xytext=(0, 8),
                     ha='center', fontsize=8)
axes[1].set_ylabel("Conversion Rate (%)")
axes[1].set_title("Monthly Overall Conversion Rate Trend")
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(0, max(cohort_stats["Conversion_Rate"])*130)

plt.tight_layout()
plt.savefig("cohort_conversion.png", bbox_inches='tight', dpi=110)
plt.show()

## Cohort Stage Heatmap

In [ ]:
cohort_stage = pd.DataFrame()
for cm in COHORTS:
    sub = df[df["cohort_month"] == cm]
    total = len(sub)
    row = {s: (sub["funnel_stage"] >= idx).sum() / total for idx, s in enumerate(STAGES)}
    row["cohort_month"] = cm
    cohort_stage = pd.concat([cohort_stage, pd.DataFrame([row])], ignore_index=True)

cohort_stage = cohort_stage.set_index("cohort_month")[STAGES]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cohort_stage * 100, annot=True, fmt=".1f", cmap="YlGnBu",
            linewidths=0.4, ax=ax, cbar_kws={"label": "% Reached Stage"})
ax.set_title("Cohort Stage Retention Heatmap (%)")
ax.set_xlabel("Funnel Stage")
ax.set_ylabel("Cohort Month")
plt.tight_layout()
plt.savefig("cohort_stage_heatmap.png", bbox_inches='tight', dpi=110)
plt.show()

## Statistical Checks

These checks test whether conversion differs materially by acquisition channel and whether converted revenue distributions vary across channels.

In [ ]:
from scipy import stats
conversion_table = pd.crosstab(df['channel'], df['converted'])
chi2_stat, chi2_p, _, _ = stats.chi2_contingency(conversion_table)
converted_revenue = df[df['converted']].groupby('channel')['revenue'].values
not_converted_revenue = df[~df['converted']].groupby('channel')['revenue'].apply(list)
channel_revenues = [df[df['channel']==ch]['revenue'].values for ch in df['channel'].unique()]
h_stat, h_p = stats.kruskal(*[g for g in channel_revenues if len(g) > 1])
print(f'Chi-square (channel vs conversion): chi2={chi2_stat:.3f}, p={chi2_p:.2e}')
print(f"Kruskal-Wallis (revenue across channels): H={h_stat:.3f}, p={h_p:.2e}")

## Revenue Leakage Analysis

In [ ]:
# How much revenue was left on the table at each stage?
# Assume every non-converted lead had an equal random opportunity equal to the average converted revenue
avg_converted_revenue = df[df["converted"]]["revenue"].mean()

leakage = []
for idx, s in enumerate(STAGES[1:], start=1):   # leads that stopped at stage idx
    dropped = df[df["funnel_stage"] == idx - 1].shape[0]   # stopped BEFORE this stage
    if idx == 1:
        dropped = (df["funnel_stage"] == 0).sum()
    else:
        dropped = (df["funnel_stage"] == idx - 1).sum()
    potential_loss = dropped * avg_converted_revenue
    leakage.append({"From_Stage": STAGES[idx-1], "To_Stage": s, "Leads_Lost": dropped, "Est_Revenue_Lost": potential_loss})

leakage_df = pd.DataFrame(leakage)
total_leak  = leakage_df["Est_Revenue_Lost"].sum()
leakage_df["Pct_Of_Total_Leakage"] = leakage_df["Est_Revenue_Lost"] / total_leak

print(f"Average revenue per converted lead: ${avg_converted_revenue:,.2f}")
print(f"\nEstimated revenue lost by stage transition:")
print(leakage_df.to_string(index=False))
print(f"\nTotal estimated unrealised revenue: ${total_leak:,.0f}")

## Visual Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
color_list = ["#e74c3c" if v == leakage_df["Est_Revenue_Lost"].max() else "#f5b7b1" for v in leakage_df["Est_Revenue_Lost"]]
bars = ax.bar(leakage_df["From_Stage"] + " → " + leakage_df["To_Stage"],
              leakage_df["Est_Revenue_Lost"] / 1e6, color=color_list, edgecolor='white')
for bar, val, pct in zip(bars, leakage_df["Est_Revenue_Lost"], leakage_df["Pct_Of_Total_Leakage"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'${val/1e6:.2f}M\n({pct:.0%})', ha='center', va='bottom', fontsize=9)
ax.set_ylabel("Estimated Revenue Lost ($ Millions)")
ax.set_title("Revenue Leakage by Funnel Stage Transition\n(red = highest leakage transition)")
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig("revenue_leakage.png", bbox_inches='tight', dpi=110)
plt.show()

## Channel Stage Waterfall: Where Each Channel Loses Leads

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax_idx, ch in enumerate(CHANNELS):
    sub = df[df["channel"] == ch]
    total = len(sub)
    reached = [(sub["funnel_stage"] >= s).sum() for s in range(N_STAGES)]
    dropped = [reached[s] - reached[s+1] if s < N_STAGES-1 else 0 for s in range(N_STAGES)]
    x = range(N_STAGES)
    axes[ax_idx].bar(STAGES, reached, color="#85c1e9", edgecolor='white', label="Retained")
    axes[ax_idx].bar(STAGES, dropped, bottom=reached, color="#e74c3c", edgecolor='white',
                     alpha=0.55, label="Dropped after" if ax_idx == 0 else "")
    axes[ax_idx].set_title(f"Channel: {ch}")
    axes[ax_idx].set_ylabel("Leads")
    axes[ax_idx].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    if ax_idx == 0:
        axes[ax_idx].legend(fontsize=8)

plt.suptitle("Funnel Waterfall by Channel", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("channel_waterfall.png", bbox_inches='tight', dpi=110)
plt.show()

## Key Metrics Summary

In [ ]:
total_leads     = len(df)
total_converted = df["converted"].sum()
overall_conv    = total_converted / total_leads
total_revenue   = df["revenue"].sum()
avg_rev_conv    = df[df["converted"]]["revenue"].mean()
best_channel    = channel_conv.iloc[0]["channel"]
worst_channel   = channel_conv.iloc[-1]["channel"]
best_region     = region_conv.iloc[0]["region"]

print("=" * 50)
print("  SALES FUNNEL — KEY METRICS SUMMARY")
print("=" * 50)
print(f"  Total Leads Entered:       {total_leads:>8,}")
print(f"  Total Converted:           {total_converted:>8,}")
print(f"  Overall Conversion Rate:   {overall_conv:>7.2%}")
print(f"  Total Revenue:             ${total_revenue:>10,.0f}")
print(f"  Avg Revenue per Conv Lead: ${avg_rev_conv:>9,.2f}")
print(f"  Best Channel:              {best_channel}")
print(f"  Worst Channel:             {worst_channel}")
print(f"  Best Region:               {best_region}")
print(f"  Highest Drop-Off Stage:    {abs_worst}")
print("=" * 50)

## Key Findings

### 1. Biggest Leakage Stage
The stage causing the most absolute lead loss is **Awareness → Interest**, which is unsurprising as
initial bounce rates are highest at the top of funnel. However the **highest relative drop-off rate**
often occurs deeper (e.g. Consideration → Intent), indicating friction in the decision phase —
typically solved by better product content, comparison pages, or trial offers.

### 2. Channel Prioritisation
**Referral** outperforms all channels on both conversion rate and average revenue per lead.
- Action: Invest in a formal referral incentive programme (e.g. 10–15 % credit per referred conversion).
- **Paid** has the lowest conversion rate despite the highest lead volume — look for audience quality issues or landing-page mismatches.

### 3. Device Optimisation
**Desktop** converts at a materially higher rate than Mobile.
- Action: Audit the mobile checkout and form flows. A/B test a mobile-first redesign of the two highest-drop-off stage pages.

### 4. Regional Gaps
**North** consistently outperforms other regions. The **South** lags.
- Action: Investigate pricing elasticity, local competition, or channel mix imbalance in the South.

### 5. Cohort Trend
If conversion rates decline across monthly cohorts it signals lead-quality degradation from scaling paid spend.
Implement cohort-level CAC monitoring and set a minimum acceptable conversion rate gate (e.g. > X% from paid).

### 6. Revenue Leakage Priority
Direct budget towards the transition that loses the most estimated revenue.
Even a 5 % uplift at the highest-leakage transition typically delivers more ROI than a 15 % improvement at a
low-volume later stage.

## Recommendations / Next Steps

| Priority | Initiative | Target Stage | Expected Impact |
|---|---|---|---|
| 1 | Referral programme formalisation | All stages | +10–15 % overall conversion |
| 2 | Mobile UX redesign (checkout + form) | Intent → Purchase | +5–8 % mobile conversion |
| 3 | Paid audience quality review | Awareness → Interest | −15 % wasted spend |
| 4 | South region localised campaign | All stages | Close ~5 % regional gap |
| 5 | Decision-stage content (comparison, trial) | Consideration → Intent | +6–10 % consideration conv |
| 6 | Cohort CAC monitoring dashboard | All channels | Catch quality decay early |

**Quick wins** (< 2 weeks): implement referral tracking, set up cohort CAC report, mobile CTA copy test.  
**Mid-term** (1–3 months): full mobile checkout redesign, paid audience segmentation overhaul.  
**Long-term** (Q3+): regional pricing strategy, personalised nurture sequences for Intent-stage leads.

## Common Mistakes
- Treating synthetic funnel rates as production truth instead of a reproducible example.
- Looking only at final conversion instead of stage-by-stage leakage.
- Comparing channel counts without normalising by lead volume.
- Ignoring cohort effects when diagnosing funnel performance.
- Treating all revenue leakage as a marketing problem rather than stage-friction problem.

## Mini Challenge / Exercises
1. Improve mobile conversion by 5 percentage points and recompute the funnel.
2. Reweight the acquisition-channel mix and compare leakage by stage.
3. Identify the weakest cohort and estimate recovered revenue if its purchase rate matched the median cohort.
4. Add a lead-score variable to the synthetic data and repeat the segmentation analysis.
5. Build a small what-if table for stage-improvement scenarios across channels.

## Limitations

1. **Synthetic data**: All figures are generated from probabilistic parameters. Conclusions are directionally
   meaningful but require validation against a real CRM or analytics export before driving investment decisions.
2. **Independent stage probabilities**: The simulation assumes each stage transition is conditionally independent.
   Real funnels exhibit lead-quality correlation across stages (good leads advance faster everywhere).
3. **No time-in-stage modelling**: Velocity (hours/days per stage) is omitted. Slow-moving leads in
   Consideration may convert but at a different LTV profile.
4. **Device attribution**: A single device is assigned per lead; real users use multiple devices, inflating
   Mobile-attributed drop-off estimates.
5. **Revenue model**: Revenue is normally distributed per channel. Real distributions are typically heavy-tailed
   (Pareto); a few large deals dominate. Adjust the generation model with a log-normal if modelling enterprise sales.

## Mini-Challenge (Extend This Notebook)
- Swap in a real CRM export (Salesforce, HubSpot) to make the charts production-grade.
- Add a **time-to-convert** column and plot stage velocity heatmaps.
- Model the funnel with a simple Markov chain to derive long-run conversion probabilities.
- Build a Plotly/Dash interactive funnel widget for stakeholder self-service.

## Final Summary / Key Takeaways

This notebook delivered a complete sales funnel drop-off analysis across five funnel stages using a
12-month synthetic lead dataset with realistic channel, region, and device segmentation.

**Key findings**:
- Awareness → Interest is the highest-volume leakage point; Consideration → Intent is often the highest rate leakage.
- Referral is the highest-converting channel; Paid has the weakest funnel efficiency.
- Desktop users convert materially better than Mobile — a direct UX optimisation opportunity.
- North outperforms South; targeted regional campaigns are warranted.
- Cohort analysis allows early detection of lead-quality decay before it harms revenue.

**All visualisations have been saved as PNG files in the working directory.**